**1.3 Kod Hücresi #1 — Excel’i okut ve ilk satırları göster**

In [19]:
import pandas as pd

df = pd.read_excel("v1_f_IT_Department_Sorunlar_ (1).xlsx")
df.head()


,id,subject,body,label
0,698,NaN,"Merhaba,\n\nİK portalına ve ERP sistemine girm...",NaN
1,2435,NaN,"Merhaba,\n\nİK portalına ve ERP sistemine girm...",NaN
2,1789,NaN,"Selamlar,\n\nVPN'e bağlanmaya çalışırken sürek...",NaN
3,2045,NaN,"Merhaba,\n\nSAP'a girmeye çalıştığımda sunucu ...",NaN
4,156,NaN,"Selamlar,\n\nEkibe yeni bir arkadaşımız katıld...",NaN


**Regex ile Ön Sınıflandırma (HATA / İSTEK / BELİRSİZ)**

Senin Excel’inde sütunlar id, subject, body, label şeklinde. Çoğu satırda subject boş, ama body dolu. Bu yüzden sınıflandırmayı “subject varsa subject, yoksa body” üzerinden yapacağız.

**2.1 Kod Hücresi #2 — Regex kuralları + yeni sütunlar**

In [20]:
import pandas as pd
import re

# =========================
# ADIM 2.0 — subject varsa subject, yoksa body'den text üret
# =========================
def build_text(row):
    subj = row.get("subject", "")
    body = row.get("body", "")
    if pd.notna(subj) and str(subj).strip() != "":
        return str(subj).strip()
    if pd.notna(body) and str(body).strip() != "":
        return str(body).strip()[:400]   # çok uzunsa kırp
    return ""

df["text"] = df.apply(build_text, axis=1)

# =========================
# ADIM 2.1 — Regex desenleri (hocanın ipucu aynen)
# =========================
HATA_REGEX  = re.compile(r"(sorun|hata|çalışmıyor|ulaşılamıyor|problem)", re.IGNORECASE)
ISTEK_REGEX = re.compile(r"(talep|istek|ekleme|kurulum|yeni)", re.IGNORECASE)

# =========================
# ADIM 2.2 — Regex ile sadece 2 ana grup: HATA / İSTEK
# Regex yakalayamazsa boş ("") bırakıyoruz
# =========================
def regex_ana_grup(text: str) -> str:
    t = (text or "").lower()
    if HATA_REGEX.search(t):
        return "HATA"
    if ISTEK_REGEX.search(t):
        return "İSTEK"
    return ""   # çözülemedi

df["regex_ana_grup"] = df["text"].apply(regex_ana_grup)
df["regex_cozulemedi_mi"] = df["regex_ana_grup"].eq("")

# =========================
# ADIM 2.3 — Kontrol çıktıları
# =========================
print("Regex dağılımı (HATA / İSTEK / COZULEMEDI):")
print(df["regex_ana_grup"].replace("", "COZULEMEDI").value_counts())

print("\nÖrnek 10 satır:")
display(df[["id", "subject", "text", "regex_ana_grup", "regex_cozulemedi_mi"]].sample(10, random_state=42))


Regex dağılımı (HATA / İSTEK / COZULEMEDI):
regex_ana_grup
COZULEMEDI    2321
HATA           559
İSTEK          120
Name: count, dtype: int64

Örnek 10 satır:


,id,subject,text,regex_ana_grup,regex_cozulemedi_mi
1801,340,NaN,"Merhaba,\n\nİşlerimi yürütebilmem için ilgili ...",,True
1190,1604,NaN,"Merhaba,\n\nİK portalına ve ERP sistemine girm...",,True
1817,331,NaN,"Merhaba,\n\nİşlerimi yürütebilmem için ilgili ...",,True
251,2247,NaN,"Selamlar,\n\nVPN'e bağlanmaya çalışırken sürek...",HATA,False
2505,41,NaN,"Merhaba,\n\nİşlerimi yürütebilmem için ilgili ...",,True
1117,1953,NaN,"Selamlar,\n\nSistemde bir şeyler ters gidiyor,...",,True
1411,2195,NaN,"Selamlar,\n\nSistemde bir şeyler ters gidiyor,...",,True
2113,904,NaN,"Selamlar,\n\nVPN'e bağlanmaya çalışırken sürek...",HATA,False
408,2974,NaN,"Merhaba,\n\nİK portalına ve ERP sistemine girm...",,True
2579,1725,NaN,"Selamlar,\n\nVPN'e bağlanmaya çalışırken sürek...",HATA,False


**BELİRSİZ olanları “LLM’e gönderme” ve 6 kategoriye ayırma**

Excel’inde subject tamamen boş, label da tamamen boş; yani sınıflandırmayı gerçekçi şekilde body üzerinden yapıyoruz.
Bu adım, sınav dokümanındaki “Regex çözemeyince AI’a gönder” + “6 kategori” kısmını karşılar.

In [23]:
import pandas as pd

# Prompt şablonu (6 kategori)
LLM_PROMPT_TEMPLATE = """Görev: Aşağıdaki IT Help Desk e-posta metnini TAM OLARAK 1 kategoriye sınıflandır.
SADECE kategori adını döndür (açıklama yazma).

Kategoriler:
- Bağlantı
- Donanım
- Erişim
- Yazılım
- Kullanıcı Yönetimi
- Raporlama

Kurallar:
- VPN, bağlantı, ağ, internet, sunucu, ulaşılamıyor -> Bağlantı
- laptop, bilgisayar, ekran, klavye, mouse, donanım -> Donanım
- erişim, yetki, izin, kullanıcı adı, şifre, giriş yapamıyorum -> Erişim
- portal/ERP/SAP/CRM, yazılım, uygulama, kurulum, hata mesajı, çöküyor, çalışmıyor -> Yazılım
- yeni kullanıcı, hesap açma, kullanıcı oluşturma -> Kullanıcı Yönetimi
- rapor, çıktı, raporlama -> Raporlama

Metin: {text}
"""

# İyileştirilmiş mock sınıflandırma
def mock_llm_classify(text: str) -> str:
    t = (text or "").lower()

    # Öncelik 1: giriş/şifre/yetki -> Erişim
    if any(k in t for k in [
        "giremiyorum","giriş","login","oturum","şifre","parola","password","kullanıcı adı","yetki","izin","erişim"
    ]):
        return "Erişim"

    # Öncelik 2: portal/erp/sap/crm -> Yazılım
    if any(k in t for k in ["portal","erp","sap","crm","uygulama","aplikasyon","program","sistem"]):
        return "Yazılım"

    # Bağlantı
    if any(k in t for k in ["vpn","bağlantı","ağ","internet","sunucu","network","ping","timeout","ulaşılamıyor"]):
        return "Bağlantı"

    # Donanım
    if any(k in t for k in ["donanım","laptop","bilgisayar","pc","ekran","klavye","mouse","printer","yazıcı","kulaklık"]):
        return "Donanım"

    # Raporlama
    if any(k in t for k in ["rapor","raporlama","çıktı","export","dışa aktar"]):
        return "Raporlama"

    # Kullanıcı Yönetimi
    if any(k in t for k in ["yeni kullanıcı","kullanıcı oluştur","hesap aç","account create","user create"]):
        return "Kullanıcı Yönetimi"

    return "Yazılım"


# ✅ Yeni routing: regex_cozulemedi_mi True ise LLM
def final_classify(row):
    text = row["text"]

    if row["regex_cozulemedi_mi"] == True:
        prompt = LLM_PROMPT_TEMPLATE.format(text=text[:400])
        category = mock_llm_classify(text)
        return pd.Series([category, "LLM", prompt])
    else:
        category = mock_llm_classify(text)
        return pd.Series([category, "Regex", ""])

df[["nihai_kategori", "route", "llm_prompt"]] = df.apply(final_classify, axis=1)

print("Route dağılımı:")
print(df["route"].value_counts())

print("\nNihai kategori dağılımı:")
print(df["nihai_kategori"].value_counts())

display(df[["id","text","regex_ana_grup","regex_cozulemedi_mi","route","nihai_kategori"]].head(10))


Route dağılımı:
route
LLM      2321
Regex     679
Name: count, dtype: int64

Nihai kategori dağılımı:
nihai_kategori
Erişim     2218
Yazılım     782
Name: count, dtype: int64


,id,text,regex_ana_grup,regex_cozulemedi_mi,route,nihai_kategori
0,698,"Merhaba,\n\nİK portalına ve ERP sistemine girm...",,True,LLM,Erişim
1,2435,"Merhaba,\n\nİK portalına ve ERP sistemine girm...",,True,LLM,Erişim
2,1789,"Selamlar,\n\nVPN'e bağlanmaya çalışırken sürek...",HATA,False,Regex,Yazılım
3,2045,"Merhaba,\n\nSAP'a girmeye çalıştığımda sunucu ...",,True,LLM,Erişim
4,156,"Selamlar,\n\nEkibe yeni bir arkadaşımız katıld...",İSTEK,False,Regex,Erişim
5,32,"Selam,\n\nBeni ilgili mail grubuna ekleyebilir...",,True,LLM,Erişim
6,836,"Selamlar,\n\nVPN'e bağlanmaya çalışırken sürek...",HATA,False,Regex,Yazılım
7,1859,"Selam,\n\nUygulamayı açtığımda direkt kapanıyo...",,True,LLM,Yazılım
8,487,"Merhaba,\n\nYeni başlayan personel için CRM si...",İSTEK,False,Regex,Erişim
9,1544,"Merhaba,\n\nİK portalına ve ERP sistemine girm...",,True,LLM,Erişim


**3.2 Prompt**

In [24]:
samples = df[df["route"] == "LLM"].sample(2, random_state=1)
for _, row in samples.iterrows():
    print("="*80)
    print("ID:", row["id"], "| Nihai:", row["nihai_kategori"])
    print(row["llm_prompt"][:1200])


ID: 1036 | Nihai: Erişim
Görev: Aşağıdaki IT Help Desk e-posta metnini TAM OLARAK 1 kategoriye sınıflandır.
SADECE kategori adını döndür (açıklama yazma).

Kategoriler:
- Bağlantı
- Donanım
- Erişim
- Yazılım
- Kullanıcı Yönetimi
- Raporlama

Kurallar:
- VPN, bağlantı, ağ, internet, sunucu, ulaşılamıyor -> Bağlantı
- laptop, bilgisayar, ekran, klavye, mouse, donanım -> Donanım
- erişim, yetki, izin, kullanıcı adı, şifre, giriş yapamıyorum -> Erişim
- portal/ERP/SAP/CRM, yazılım, uygulama, kurulum, hata mesajı, çöküyor, çalışmıyor -> Yazılım
- yeni kullanıcı, hesap açma, kullanıcı oluşturma -> Kullanıcı Yönetimi
- rapor, çıktı, raporlama -> Raporlama

Metin: Merhaba,

İK portalına ve ERP sistemine girmeye çalıştığımda sayfa bir türlü yüklenmiyor. Bazen tam işlem yapacakken ekran donuyor ve sistemden atıyor. Giriş yaparken de çok bekletiyor, bir türlü ilerleyemiyorum. Sistemlerde genel bir yavaşlık mı var yoksa benim kullanıcımla mı ilgili bir durum? Bakabilirseniz sevinirim.

Kolay gels

**Sonuçları Excel’e yazdırma**

Her kayıt için Regex sonucu + Route (Regex/LLM) + Nihai kategori ve prompt.

**4.1 Kod Hücresi #4 — Teslim Excel’ini üret**

In [26]:
import pandas as pd

# ADIM 4 — Teslim Excel dosyasını üret
deliver_cols = [
    "id",
    "subject",
    "text",
    "regex_ana_grup",
    "regex_cozulemedi_mi",
    "route",
    "nihai_kategori",
    "llm_prompt"
]

# Bazı excel dosyalarında subject olmayabilir; güvenli olsun diye kontrol:
available_cols = [c for c in deliver_cols if c in df.columns]

out_df = df[available_cols].copy()

output_file = "IT_HelpDesk_Siniflandirma_Sonuc.xlsx"
out_df.to_excel(output_file, index=False)

print("Çıktı dosyası oluşturuldu:", output_file)
print("Yazılan kolonlar:", available_cols)

# İlk 15 satırı göster
display(out_df.head(15))

# (Opsiyonel) Özet sayımlar
print("\nRegex ana grup dağılımı:")
print(out_df["regex_ana_grup"].replace("", "COZULEMEDI").value_counts())

print("\nRoute dağılımı:")
print(out_df["route"].value_counts())

print("\nNihai kategori dağılımı:")
print(out_df["nihai_kategori"].value_counts())


Çıktı dosyası oluşturuldu: IT_HelpDesk_Siniflandirma_Sonuc.xlsx
Yazılan kolonlar: ['id', 'subject', 'text', 'regex_ana_grup', 'regex_cozulemedi_mi', 'route', 'nihai_kategori', 'llm_prompt']


,id,subject,text,regex_ana_grup,regex_cozulemedi_mi,route,nihai_kategori,llm_prompt
0,698,NaN,"Merhaba,\n\nİK portalına ve ERP sistemine girm...",,True,LLM,Erişim,Görev: Aşağıdaki IT Help Desk e-posta metnini ...
1,2435,NaN,"Merhaba,\n\nİK portalına ve ERP sistemine girm...",,True,LLM,Erişim,Görev: Aşağıdaki IT Help Desk e-posta metnini ...
2,1789,NaN,"Selamlar,\n\nVPN'e bağlanmaya çalışırken sürek...",HATA,False,Regex,Yazılım,
3,2045,NaN,"Merhaba,\n\nSAP'a girmeye çalıştığımda sunucu ...",,True,LLM,Erişim,Görev: Aşağıdaki IT Help Desk e-posta metnini ...
4,156,NaN,"Selamlar,\n\nEkibe yeni bir arkadaşımız katıld...",İSTEK,False,Regex,Erişim,
5,32,NaN,"Selam,\n\nBeni ilgili mail grubuna ekleyebilir...",,True,LLM,Erişim,Görev: Aşağıdaki IT Help Desk e-posta metnini ...
6,836,NaN,"Selamlar,\n\nVPN'e bağlanmaya çalışırken sürek...",HATA,False,Regex,Yazılım,
7,1859,NaN,"Selam,\n\nUygulamayı açtığımda direkt kapanıyo...",,True,LLM,Yazılım,Görev: Aşağıdaki IT Help Desk e-posta metnini ...
8,487,NaN,"Merhaba,\n\nYeni başlayan personel için CRM si...",İSTEK,False,Regex,Erişim,
9,1544,NaN,"Merhaba,\n\nİK portalına ve ERP sistemine girm...",,True,LLM,Erişim,Görev: Aşağıdaki IT Help Desk e-posta metnini ...



Regex ana grup dağılımı:
regex_ana_grup
COZULEMEDI    2321
HATA           559
İSTEK          120
Name: count, dtype: int64

Route dağılımı:
route
LLM      2321
Regex     679
Name: count, dtype: int64

Nihai kategori dağılımı:
nihai_kategori
Erişim     2218
Yazılım     782
Name: count, dtype: int64
